# **Goal**

The purpose of these notebooks is to present some initial experiments to motivate model selection for an interactive app that allows users to experiment with small-medium language model (100M - 1B range).

####**Qwen3-0.6B-Base**

- <u>Total params</u>: 600M. This is the total number of learned scalar values in the entire model: weights, biases, embeddings, output projection, etc.
- <u>Non-embdedding params</u>: 440M. These are the learned parameters in the transformer machinery itself, excluding the token embedding and output vocabulary matrices.
- <u>Number of layers</u>: 28
- <u>Model dimension</u>: 1024. I.e.: Every token is represented internally by a vector of length 1024.
- <u>MLP units per layer:</u> 3072
- <u>Attention Heads</u>: 16 for Q and 8 for KV
- <u>Vocabulary size:</u> 151,936. Amount of tokens in vocabulary.
- <u>Max context window</u>: 32,768 tokens.

Load model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B-Base")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B-Base", device_map="auto")
messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Who are you? ⚇ ⚇ ⚇
Who are you? ⚇ ⚇ ⚇
Who are you? ⚇ ⚇ �


In [ ]:
print(model.config)

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_w

Prompt experiments. Can play with the following generation parameters:

- `max_new_tokens:` Intenger determining length of response excluding initial prompt.
- `do_sample:` Boolean determining greedy/non-greedy sampling.
- `temperature:` Low temp -> sharp distribution; high temp -> flattened distribution. I.e., how predictable or conservative the model is when sampling tokens.
- `top_k:` Only consider the \(k\) highest-scoring tokens. Default = 50.
- `top_p:` Float 0-1.0. Keep the smallest set of most likely tokens whose probabilities together account for at least \(p\) of the probability mass. Default = 1.
- `repetition_penalty:` How much repeated tokens are discouraged. Default = 1.0 means no penalty.

In [ ]:
messages = [
    {"role": "user", "content": "How much do you like ice cream?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

How much do you like ice cream? ⍼ ⍼ ⍼ ⍼ ⍼ ⍼ ⍼ ⍼ ⍼ ⍼ ⍼ ⍼ ⍼ ⍼


In [ ]:
inputs = tokenizer("The capital of France is", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

'The capital of France is Paris, and the capital of Germany is Berlin. If the distance between Paris and Berlin is 1000 km, what is the distance between the capital of France and the capital of Germany? The distance between the capital of France and the capital of Germany is the same as the distance between Paris and Berlin, which is 1000 km.<|endoftext|>'

In [ ]:
inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
)

print(tokenizer.decode(output[0]))

The capital of France is Paris, France. What's the capital of France? The capital of France is Paris.<|endoftext|>


In [ ]:
inputs = tokenizer("What is love?", return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(tokenizer.decode(output[0], skip_special_tokens=True))

Input tokens: tensor([3838,  374, 2948,   30])
Output tokens: tensor([ 2585,   646,   582,  2225,  1039, 14269,   323,  3590, 11229,   311,
          387,   264,  1661,    11, 29498,  1697,  1939, 28251, 17601,  5538,
        36705,   476, 32364,  1948,  1378,  7775,    13,  1084,   646, 19101,
          504,  6961,  3645,   320, 30053,     8,   714,  3545,  2239,  7797,
          432,   979,  1052,   525,  6094,  2750,    11, 20799,    11, 16005])

What is love? How can we develop our emotional and social intelligence to be a good, caring person?

Love involves deep affection or attraction between two individuals. It can stem from physical contact (love) but often extends beyond it when there are shared values, beliefs, hopes


Here we are just printing the internal shape of one layer of the model.


In [ ]:
print(model.model.layers[0].mlp)

Qwen3MLP(
  (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
  (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
  (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
  (act_fn): SiLUActivation()
)


In [ ]:
model.config.intermediate_size

3072

The input vector representation is 1024-dimensional. The intermediate dimension is 3072. Note that the Qwen models use more sophisticated gated MLPS.

There are 2 separate sets of 3072 MLP units that receive the input. Each of these produces a a transformed and expanded 3072-d vector called the gate projection and the up projection. The gate projection goes through activation using SiLU (Sigmoid Linear Unit). Then this activated projection is multiplied element-wise with the up-projection, and these new values are transformed back into a 1024-d representation by another learned matrix. This representation will continue through the transformer block and subsequent layers.

Note: $\text{SiLU}(x) = x ⋅ σ(x)$, where $σ(x)$ is the sigmoid function $\frac{1}{1+e^{-x}}$.

**MLP Unit Ablation Experiments**

In [ ]:
import torch

def add_mlp_ablation(model, layer_idx, proportion, seed=42):

    # MLP in layer layer_idx
    mlp = model.model.layers[layer_idx].mlp

    num_units = model.config.intermediate_size
    num_ablate = int(proportion * num_units)

    # Seed RNG
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Pick 'num_ablate' indexes for MLP units at random
    indices = torch.randperm(
        num_units,
        generator=generator
    )[:num_ablate]

    def ablation_hook(module, inputs):
        x = inputs[0].clone()
        indices_device = indices.to(x.device)
        x[..., indices_device] = 0
        return (x,) + inputs[1:]

    handle = mlp.down_proj.register_forward_pre_hook(ablation_hook)

    return handle, indices, layer_idx, proportion

Here we try zeroing the outputs of different proportions of the first (0th) layer. Notice gradual degradation in output coherence.

Baseline

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed activations in {proportion}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Input tokens: tensor([ 785, 6722,  315, 9625,  374])
Output tokens: tensor([12095,    11,   323,   279,  7772,  3283,   304,  4505,    13,   576,
         8585,   525,  3881,   369,   862,  2948,   315, 13078,   320, 34946,
        83957,   701, 35005,  3489, 43197,     1,  3607,     8,   448,   264,
         3746, 24654,   389,  7640,   438,  1632,   438, 13041, 25779,  1075,
          330,  5368, 48263,   326,     6,  1310,   497,   892,  3363, 52305])

Zeroed activations in 0.0% of units in layer 0
The capital of France is Paris, and the largest city in Europe. The French are known for their love of wine (especially Bordeaux), cuisine ("French" food) with a strong emphasis on fish as well as meat dishes like "poisson l'air", which means seafood


$25\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.25,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 785, 6722,  315, 9625,  374])
Outputs: tensor([12095,    13,   576,  7772,  3283,   304,   279,  7513,  9145,    11,
          432,   594,  1083,   825,   315,  4505,   748,  1429,  5411, 29970,
        33105,   624, 59604,   702,   264,  9080,  3840,   323,  7674,   429,
          646,   387, 35031,   553,  1803,   476,   389,  4478,    13,  1084,
        37019,  1657, 50577,    11,  1947, 42554,    11, 15556,    11, 73571])

Zeroed activations in 25.0% of units in layer 0
The capital of France is Paris. The largest city in the European Union, it's also one of Europe’s most popular tourist destinations.
Paris has a rich history and culture that can be explored by car or on foot. It boasts many museums, art galleries, restaurants, cafes


$50\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.5,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 785, 6722,  315, 9625,  374])
Outputs: tensor([12095,    13,   576,  8585,  3409,   369,   330,  8926,     1,  4041,
          504,   279, 19458,  3283,    11,   323,   432,  3363,   264,  1992,
         1380,  1251,  3887,  3786,   304,   825,  4752,   476,  1874,   624,
         3838,  1558,   279,   829,  3076,  5267, 59604,   320, 43197,    25,
         4270, 23422,     8,   374,   458, 13833, 12751, 40936,  7407,   389])

Zeroed activations in 50.0% of units in layer 0
The capital of France is Paris. The French word for "city" comes from the Latin city, and it means a place where people live together in one building or group.
What does the name mean?
Paris (French: París) is an ancient Roman colony located on


$75\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.75,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 785, 6722,  315, 9625,  374])
Outputs: tensor([  279,  7772,  3283,   304,   892,  3146,  5267,  3838,   525,  1045,
         2513,   429,   498,   646,   653,   311,  1281,   697,  2272,   803,
         7040,    30,  3555,   594,   264,  1661,  1616,   369,  3259,   432,
         2464, 88983,  1939, 49000,    11,   438,  1181,   829, 13230,    11,
          702,  1657,  9720,    13,   576,  8538,   825,   553,  3041,  4436])

Zeroed activations in 75.0% of units in layer 0
The capital of France is the largest city in which country?
What are some things that you can do to make your life more interesting? What's a good way for making it funnier?

France, as its name suggests, has many cities. The biggest one by far isn


$100\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=1.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 785, 6722,  315, 9625,  374])
Outputs: tensor([  198,    32,    13, 12095,   425,    13,  7148,   356,    13, 19846,
          422,    13, 21718,   271, 14582,   220,    16,    15,   481,   576,
        18374,  4311,  4940,  9625,  2160,   362,     8, 59604,    33,  1214,
         6474,    34,  1785,   261,  3732,    35,  2013,   635, 15846,   220,
           17,    20, 15920, 14106, 11443,  2308,  5055, 22666,    30,   362])

Zeroed activations in 100.0% of units in layer 0
The capital of France is
A. Paris B. London C. Berlin D. Rome

Question 10 - The Capital City Of France Is A)ParisB.LondonC.BerlinD.Rome Question 25 Which Country Has No National Flag? A


**Attention Head Ablation Experiments**

First let's check the attention heads

In [ ]:
print(model.model.layers[0])

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
    (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
)


We see that the attention mechanism expects a 1024d input (the token embedding or output from the previous block). It then computes a 2048-d vector q_proj, and two 1024-d vectors, k_proj and v_proj. This corresponds to calculating its query, key, and value vectors. Note that there are are 16 Q-heads and 8-KV heads, so each key/value head is shared by two query heads. After computing attention scores using these 3 vectors, the attention mechanism produces 16 query-head outputs, each with dimension 128. This 2048 output goes into the o-projection, a learned projection from the concatenated attention head contributions, to the final attention contribution with dimension 1024.

In [ ]:
print(model.model.layers[0].self_attn.head_dim)

128


In [ ]:
model.config.num_hidden_layers

28

In [ ]:
model.config.hidden_size

1024

In [ ]:
model.config.num_attention_heads

16

Attention head ablation function

In [ ]:
def add_head_ablation(model, head_indices, layer_idx):

    #Number of hidden layers
    num_layers = model.config.num_hidden_layers
    #Number of Query attention heads
    num_heads = model.config.num_attention_heads

    if not 0 <= layer_idx < num_layers:
        raise ValueError(f"layer_idx must be between 0 and {num_layers - 1}")
    for head_idx in head_indices:
        if not 0 <= head_idx < num_heads:
            raise ValueError(f"head_idx must be between 0 and {num_heads - 1}")

    #Projection dimension per attention head
    head_dim = model.model.layers[layer_idx].self_attn.head_dim

    # Load final attention module
    attention_dense = model.model.layers[layer_idx].self_attn.o_proj

    def head_ablation_hook(module, inputs):
        x = inputs[0].clone()

        for head_idx in head_indices:
            #compute indices corresponding to specific head
            start = head_idx * head_dim
            end = start + head_dim

            #Select indices in the head outputs vector corresponding to chosen head and zero them
            x[..., start:end] = 0

        return (x,) + inputs[1:]

    #Note this is a pre-hook because we want to intervene components before the o-projection forward pass
    handle = attention_dense.register_forward_pre_hook(head_ablation_hook)

    return handle, head_indices, layer_idx

Here we try zeroing the outputs of different numbers of heads of the first (0th) layer.

Remove 1 head from layer 0

In [ ]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([ 785, 6722,  315, 9625,  374])
Output tokens: tensor([12095,    13,   576,  7772,  3283,   304,   279,  3639,  4180,    11,
         1532,  4261,  4311,   702,   264,  7042,   916,   220,    23,  3526,
         1251,   323,   432,   594,  1083,  2114,   311,  1657, 11245, 59924,
         1741,   438, 10684,  5540,   624,  3838,   525,  1045,  1008,  9720,
          429,   614,  3460, 21910,  5267,  8373,  1008,  3598,  9720,   448])

Zeroed head [0] in layer 0
The capital of France is Paris. The largest city in the United States, New York City has a population over 8 million people and it's also home to many famous landmarks such as Central Park.
What are some other cities that have large populations?
Some other major cities with


Ablate 4 attention heads in layer 0

---



In [ ]:
to_ablate = int(model.config.num_attention_heads*0.25)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 785, 6722,  315, 9625,  374])
Outputs: tensor([  264,   220,   279,  3146,   429,   702,  1012, 18047,   553, 47587,
           13,   323,   432,   572,   304,  3465,   553, 47587,   624, 12903,
        34313,  1817,  3409,   389,   419, 11652,   382, 39814,    11,  1588,
          594,  1246,   498,   646, 52725,   697,  8585, 22870,  1447,     1,
          576, 18374,  2022,   369,   330, 49000,     1,   374,   279,  6995])

Zeroed head [0, 1, 2, 3] in layer 0
The capital of France is a  the country that has been founded by france. and it was in created by france.
Capitalize each word on this sentence.

Sure, here's how you can capitalize your French sentences:

" The Capitalization for "France" is the nation


Ablate 8 attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads*0.5)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 785, 6722,  315, 9625,  374])
Outputs: tensor([  752,    72,   600,  9497,  1853, 56253,   297, 18740,   264,    64,
           78,  2429,  6215,   409, 30121,    13, 39991,   294,   296, 14460,
          384,   259,  3808,    68,  8988,    84,   575,  1842,  3845,   976,
        12278,  8155,  8579,   336,   307,    76,   318,  1054,  3262,  2660,
           69,   450,    83,  1963,   295, 30072,   332,   372,  7066, 20826])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7] in layer 0
The capital of France is mei i mi di mia o meg aao demora de momento. moi d m ii e tiie tiu u et du emidimidiemidmimduempmefdetuietuuutumettutt


Ablate 12 attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads*0.75)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([ 785, 6722,  315, 9625,  374])
Output tokens: tensor([   271,   1773,    334,     63,     69, 151643])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] in layer 0
The capital of France is

。**`f


Ablate 16 (all) attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([ 785, 6722,  315, 9625,  374])
Output tokens: tensor([   552,  80519, 120075, 111572,    481,    220,   6447,   9796, 124806,
          9325,  10148,   5400,    602,    535,  35541,     66,     34,  58827,
        126325,  22097,   1307,  65501,  77762,   2252,  42967,    471, 136124,
        104679,  92307,   2359,   8656,   3511,  13552,     41,   2195,  79531,
        113999,    506,    336,     90,    284,     25,  33108,     17,    491,
           271,     46,    437,   4710,     16])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] in layer 0
The capital of France isureolated郦咏 - chadpçekoonoerryppostkalcCBaçikommplleKatxtxnartёр塘artnerleftkanpondCompJOr�跋 Oem{ =:和2ight

Oand 

1


Ablate all attention heads in layer 11

In [ ]:
to_ablate = int(model.config.num_attention_heads)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=11,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 785, 6722,  315, 9625,  374])
Outputs: tensor([12095,    13,   576,  7772,  3283,   304,   279,  3146,    11,   432,
          702,   264,  7042,   429, 35275,   220,    16,    15,  3526,  1251,
          624, 59604,   320, 43197,    25,   330,  4272,   285,   899,   572,
        18047,   438,   458,  9489, 78145,   389,  5470,  4102,    17,    20,
          339,   553,  6210, 11876,   358,   323,   806,  7403,  2160, 54167])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] in layer 11
The capital of France is Paris. The largest city in the country, it has a population that exceeds 10 million people.
Paris (French: "Paris") was founded as an independent commune on March 25th by King Louis I and his wife Isabelle


Ablate heads in every layer

Baseline

In [ ]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Baseline")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Baseline
The capital of France is Paris, and the largest city in Europe. The French are known for their love of wine (especially Bordeaux), cuisine ("French" food) with a strong emphasis on fish as well as meat dishes like "poisson l'air", which means seafood


In [ ]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed head 0 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed head 0 in all layers
The capital of France is ____.
A. Paris
B. London
C. Rome
D. Berlin
Answer:
Paris

Which country has the most universities?
A) Germany, Italy and Spain have more than 10% each in their university rankings.
B


In [ ]:
to_ablate = int(model.config.num_attention_heads*0.25)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3 in all layers
The capital of France is the only one country to that has a single passport. france,
 it's not just another word for french people and they are all French or francophiles? I don't know but i am sure you're right so far from anyone who knows


In [ ]:
to_ablate = int(model.config.num_attention_heads*0.5)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3,4,5,6,7 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3,4,5,6,7 in all layers
The capital of France isodoxgebung toenarten_node头ittal_ti_en_binding_il Taliban {* buddiesinhukkitiuien晋playiancessburgze该院hardtwiseicialhuithti,nullinaryingen\Unit bestebine tutinniehariuzzi培训班thеть\Corejualaltaught.only/manualIBUT


In [ ]:
to_ablate = int(model.config.num_attention_heads)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Output tokens: tensor([ 31627, 114797,  77753, 100541,  37272,  52261, 144958,  79697,  18326,
         62626, 104921, 102989, 136621,  99522,    609, 140832, 104191, 139819,
         13735,  67375,  87742,  92263,  92599,  15094,  45352,  73199,  63730,
         39104,  91794,  24794,  63579,  16862, 121727, 117159,  63419,  31947,
         55182,  13064,    353,   2146,   2707, 108717,  59131,  53396, 119665,
           311,  17412,    613, 117190,  55255])

Zeroed all heads in all layers
The capital of France issgiving力学 wan赏[, {%▾ptom.Sc-choice年来使其今は方面 & הוד一份 véritable_CLASSltrositesbinếusrconte><![klävenile realpathammed ../../////////////橛也不例外vie********************ceries facts *...

erties为我们 searchDataammad苣 toitionallyell已久的 Gone


In [ ]:
to_ablate = int(model.config.num_attention_heads)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Output tokens: tensor([ 31627, 114797,  77753, 100541,  37272,  52261, 144958,  79697,  18326,
         62626, 104921, 102989, 136621,  99522,    609, 140832, 104191, 139819,
         13735,  67375,  87742,  92263,  92599,  15094,  45352,  73199,  63730,
         39104,  91794,  24794,  63579,  16862, 121727, 117159,  63419,  31947,
         55182,  13064,    353,   2146,   2707, 108717,  59131,  53396, 119665,
           311,  17412,    613, 117190,  55255])

Zeroed all heads in all layers
The capital of France issgiving力学 wan赏[, {%▾ptom.Sc-choice年来使其今は方面 & הוד一份 véritable_CLASSltrositesbinếusrconte><![klävenile realpathammed ../../////////////橛也不例外vie********************ceries facts *...

erties为我们 searchDataammad苣 toitionallyell已久的 Gone
